# Running baseline models
Normal classification of diagnoses

## Setup

In [1]:
from pathlib import Path
import subprocess
import time
import warnings
import numpy as np
import pandas as pd
import sklearn
import mlflow
import dagshub
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, log_loss, roc_auc_score)
from uuid import uuid4
import os
import json

TRAIN_PATH = os.path.join(
    r"C:\Users\Hzaab\SDP-Sequential-Diagnosis",
    "afabench", "Afabench_Source", "afabench",
    "extra", "data", "misc", "ddxplus_train.csv"
)
TEST_PATH = Path("C:/Users/Hzaab/SDP-Sequential-Diagnosis/afabench/Afabench_Source/afabench/extra/data/misc/ddxplus_test.csv")
TARGET = "pathology"
version = "Fifth"
fraction = 1
seed = 42
val_size = 0.2
owner = "HindAlz"
repo = "Sequential-Diagnosis"
experiment = "sdp-sequential-diagnosis"


c:\Users\Hzaab\SDP-Sequential-Diagnosis\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_owner=owner, repo_name=repo, mlflow=True)
mlflow.set_experiment(experiment)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print("Commit:", commit)

Accessing as HindAlz

Initialized MLflow to track repo "HindAlz/Sequential-Diagnosis"

Repository HindAlz/Sequential-Diagnosis initialized!

Commit: fb860fa0424799e1807a70a5876728e47b76d798


In [3]:
import mlflow
import pandas as pd

tables = []

for experiment in mlflow.search_experiments():
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        max_results=100000
    )
    runs["experiment_name"] = experiment.name
    tables.append(runs)

results = pd.concat(tables, ignore_index=True)
results.to_csv("experiment_results.csv", index=False)

print("Exported runs:", len(results))

Exported runs: 53


## Preprocessing

In [ ]:
train_full = pd.read_csv(TRAIN_PATH)
test_full = pd.read_csv(TEST_PATH)


In [ ]:
#for non binary included
feature_columns = [c for c in train_full.columns if c != TARGET]
numeric_columns = train_full[feature_columns].select_dtypes(include=np.number).columns.tolist()
categorical_columns = train_full[feature_columns].select_dtypes(exclude=np.number).columns.tolist()


In [ ]:
#for binary only
metadata_path = os.path.join(
    r"C:\Users\Hzaab\SDP-Sequential-Diagnosis",
    "afabench",
    "Afabench_Source",
    "data",
    "release_evidences.json"
)

with open(metadata_path, encoding="utf-8-sig") as f:
    evidence_defs = json.load(f)

feature_columns = []
missing_binary = []

for evidence_id, definition in evidence_defs.items():
    if definition["data_type"] == "B":
        if evidence_id in train_full.columns:
            feature_columns.append(evidence_id)
        else:
            missing_binary.append(evidence_id)

numeric_columns = feature_columns.copy()
categorical_columns = []

print("Binary features used:", len(feature_columns))
print("Binary evidence missing from CSV:", missing_binary)



Binary features used: 207
Binary evidence missing from CSV: ['E_53']
Full rows: train=  820481 :, test=  102561 :
Sampled rows: train=  820481 :, test=  102561 :
Fit=  656384 , validation=  164097  classes= 49


In [ ]:

def sample_rows(df):
    if fraction == 1:
        return df.copy()

    selected, _ = train_test_split(df, train_size=fraction, stratify=df[TARGET], random_state=seed)
    
    return selected.copy()

train = sample_rows(train_full)
test = sample_rows(test_full)

X = train[feature_columns]
y = train[TARGET]

X_test = test[feature_columns]
y_test = test[TARGET]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=val_size, stratify=y, random_state=seed)

print("Full rows: train= ",len(train_full),":, test= ", len(test_full),":")
print("Sampled rows: train= ", len(train),":, test= ", len(test),":")
print("Fit= ", len(X_train),", validation= ", len(X_val), " classes=", y.nunique())

del train_full, test_full

In [20]:
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train = label_encoder.transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

y = label_encoder.transform(y)

## models general settings

In [ ]:
#non binary included
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING", keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]), categorical_columns),
], sparse_threshold=1.0)

In [ ]:
#binary only
preprocessor = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent",
        keep_empty_features=True
    )),
    ("scale", StandardScaler())
])

In [ ]:


models = {
    "RF": RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=seed),
    "DT": DecisionTreeClassifier(max_depth=20, min_samples_leaf=2, random_state=seed),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,), max_iter=100, batch_size="auto", random_state=seed),
    "LOGREG": LogisticRegression(max_iter=500, random_state=seed),
    "ET": ExtraTreesClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=seed),
    "DUMMY": DummyClassifier(strategy="prior", random_state=seed),
    "XGB": XGBClassifier(device="cuda", tree_method="hist", n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, max_bin=128, random_state=seed, n_jobs=2)
}


In [22]:
def evaluate(model, features, target):
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)
    classes = model.classes_

    metrics = {
        "accuracy": accuracy_score(target, predictions),
        "precision": precision_score(target, predictions, labels=classes, average="macro", zero_division=0),
        "recall": recall_score(target, predictions, labels=classes, average="macro", zero_division=0),
        "f1": f1_score(target, predictions, labels=classes, average="macro", zero_division=0),
        "loss": log_loss(target, probabilities, labels=classes),
    }
    
    if set(target) == set(classes):
        metrics["AUC"] = (
            roc_auc_score(target == classes[1], probabilities[:, 1])
            if len(classes) == 2 else
            roc_auc_score(target, probabilities, labels=classes,  multi_class="ovr", average="macro")
        )
    return {key: float(value) for key, value in metrics.items()}


In [23]:
results = []
trial_configs = {}

In [ ]:

def run_trial(model_type, trial_name="trial", **hyperparams):
    estimator = clone(models[model_type]).set_params(**hyperparams)
    actual_params = estimator.get_params(deep=False)

    model = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("classifier", estimator),
    ])

    trial_slug = "_".join(trial_name.strip().lower().split())
    trial_id = uuid4().hex[:8]

    run_name = (
        f"{version}_{model_type}_"
        f"{trial_slug}_validation_seed{actual_params.get('random_state', seed)}"
        f"_{trial_id}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            "mlflow.source.git.commit": commit,
            "commit": str(commit),
            "dataset_version": version,
            "split": "validation",
            "trial_name": trial_name,
            "metric_average": "macro",
            "loss_definition": "log_loss",
            "feature_set": "binary_evidence_only" #for non binary only
        })

        mlflow.log_params({
            "model": model_type,
            "seed": actual_params.get("random_state", seed),
            "sample_fraction": fraction,
            "validation_size": val_size,
            "target": TARGET,
            "n_features_raw": len(feature_columns),
            "n_fit": len(X_train),
            "n_eval": len(X_val),
            "sklearn_version": sklearn.__version__,
            **actual_params,  # Logs the parameters actually used.
        })

        started = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - started

        metrics = evaluate(model, X_val, y_val)
        mlflow.log_metrics({
            **metrics,
            "fit_seconds": fit_seconds,
        })

        if "AUC" not in metrics:
            mlflow.set_tag(
                "auc_note",
                "Skipped: validation does not contain every training class",
            )

        run_id = mlflow.active_run().info.run_id

    row = {
        "model": model_type,
        "trial": trial_name,
        **metrics,
        "fit_seconds": fit_seconds,
        "hyperparams": hyperparams.copy(),
        "run_id": run_id,
    }

    results.append(row)

    # Save exact settings for refitting the winning configuration later.
    trial_configs[run_id] = {
        "model_type": model_type,
        "params": actual_params.copy(),
    }

    print("Run:", run_name)
    print("Run ID:", run_id)
    display(pd.DataFrame([row]))

    return model

## models

In [25]:
xgb_model= run_trial("XGB",
    trial_name="gpu-depth6-100trees", n_estimators=100, max_depth=6, learning_rate=0.1)

🏃 View run Fifth_XGB_gpu-depth6-100trees_validation_seed42_06c5fe93 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/4d1eb4e7e343454298822a32ab50582d
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_XGB_gpu-depth6-100trees_validation_seed42_06c5fe93
Run ID: 4d1eb4e7e343454298822a32ab50582d


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,XGB,gpu-depth6-100trees,0.993291,0.993668,0.99119,0.991722,0.020131,0.9999,66.596575,"{'n_estimators': 100, 'max_depth': 6, 'learnin...",4d1eb4e7e343454298822a32ab50582d


In [26]:
rf_model= run_trial("RF",
    trial_name="200trees-depth15", n_estimators=200, max_depth=15, min_samples_leaf=2)

🏃 View run Fifth_RF_200trees-depth15_validation_seed42_3b076e2b at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/d963c09c20564519892ddaefbaf47068
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_RF_200trees-depth15_validation_seed42_3b076e2b
Run ID: d963c09c20564519892ddaefbaf47068


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,RF,200trees-depth15,0.951553,0.958307,0.877378,0.891841,1.036241,0.999426,35.281568,"{'n_estimators': 200, 'max_depth': 15, 'min_sa...",d963c09c20564519892ddaefbaf47068


In [27]:
dt_model = run_trial("DT",
    trial_name="depth10", max_depth=10, min_samples_leaf=5)

🏃 View run Fifth_DT_depth10_validation_seed42_d2823d0f at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/785c9474591746c9a25aebb2a7af30cf
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_DT_depth10_validation_seed42_d2823d0f
Run ID: 785c9474591746c9a25aebb2a7af30cf


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,DT,depth10,0.476036,0.428217,0.357585,0.357064,1.890158,0.844038,8.270944,"{'max_depth': 10, 'min_samples_leaf': 5}",785c9474591746c9a25aebb2a7af30cf


In [28]:
mlp_model = run_trial("MLP",
    trial_name="128-64", hidden_layer_sizes=(128, 64), learning_rate_init=0.001, alpha=0.001, max_iter=200)

🏃 View run Fifth_MLP_128-64_validation_seed42_9058dd40 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/7206899bc1874a52924be47c4dc52adc
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_MLP_128-64_validation_seed42_9058dd40
Run ID: 7206899bc1874a52924be47c4dc52adc


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,MLP,128-64,0.992876,0.99285,0.990178,0.990752,0.020345,0.999899,202.09824,"{'hidden_layer_sizes': (128, 64), 'learning_ra...",7206899bc1874a52924be47c4dc52adc


In [29]:
logreg_model = run_trial("LOGREG",
    trial_name="C0.1", C=0.1, max_iter=1000)

🏃 View run Fifth_LOGREG_c0.1_validation_seed42_da2baa3e at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/07d6675f8cef4266ac63f4e23f5abf0a
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_LOGREG_c0.1_validation_seed42_da2baa3e
Run ID: 07d6675f8cef4266ac63f4e23f5abf0a


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,LOGREG,C0.1,0.993102,0.994197,0.990807,0.991682,0.020027,0.9999,68.884552,"{'C': 0.1, 'max_iter': 1000}",07d6675f8cef4266ac63f4e23f5abf0a


In [30]:
et_model = run_trial(
    "ET",
    trial_name="200trees-depth20", n_estimators=200, max_depth=20)

🏃 View run Fifth_ET_200trees-depth20_validation_seed42_adb46184 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/a9a496b1422140168078c55c8c395103
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_ET_200trees-depth20_validation_seed42_adb46184
Run ID: a9a496b1422140168078c55c8c395103


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,ET,200trees-depth20,0.970018,0.962569,0.918655,0.924713,0.722542,0.999685,72.590626,"{'n_estimators': 200, 'max_depth': 20}",a9a496b1422140168078c55c8c395103


In [31]:
dummy_model = run_trial("DUMMY", trial_name="prior")

🏃 View run Fifth_DUMMY_prior_validation_seed42_77ecba30 at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0/runs/d6b54e0dd0c34364bae9e7f9802105c9
🧪 View experiment at: https://dagshub.com/HindAlz/Sequential-Diagnosis.mlflow/#/experiments/0
Run: Fifth_DUMMY_prior_validation_seed42_77ecba30
Run ID: d6b54e0dd0c34364bae9e7f9802105c9


,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,DUMMY,prior,0.062762,0.001281,0.020408,0.00241,3.730539,0.5,4.91959,{},d6b54e0dd0c34364bae9e7f9802105c9


In [32]:
leaderboard = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

display(leaderboard)

,model,trial,accuracy,precision,recall,f1,loss,AUC,fit_seconds,hyperparams,run_id
0,XGB,gpu-depth6-100trees,0.993291,0.993668,0.991190,0.991722,0.020131,0.999900,66.596575,"{'n_estimators': 100, 'max_depth': 6, 'learnin...",4d1eb4e7e343454298822a32ab50582d
1,LOGREG,C0.1,0.993102,0.994197,0.990807,0.991682,0.020027,0.999900,68.884552,"{'C': 0.1, 'max_iter': 1000}",07d6675f8cef4266ac63f4e23f5abf0a
2,MLP,128-64,0.992876,0.992850,0.990178,0.990752,0.020345,0.999899,202.098240,"{'hidden_layer_sizes': (128, 64), 'learning_ra...",7206899bc1874a52924be47c4dc52adc
3,ET,200trees-depth20,0.970018,0.962569,0.918655,0.924713,0.722542,0.999685,72.590626,"{'n_estimators': 200, 'max_depth': 20}",a9a496b1422140168078c55c8c395103
4,RF,200trees-depth15,0.951553,0.958307,0.877378,0.891841,1.036241,0.999426,35.281568,"{'n_estimators': 200, 'max_depth': 15, 'min_sa...",d963c09c20564519892ddaefbaf47068
5,DT,depth10,0.476036,0.428217,0.357585,0.357064,1.890158,0.844038,8.270944,"{'max_depth': 10, 'min_samples_leaf': 5}",785c9474591746c9a25aebb2a7af30cf
6,DUMMY,prior,0.062762,0.001281,0.020408,0.002410,3.730539,0.500000,4.919590,{},d6b54e0dd0c34364bae9e7f9802105c9


In [1]:
import os
import json

folder = r"C:\Users\Hzaab\SDP-Sequential-Diagnosis\models\gdfs_xgb_website"

with open(os.path.join(folder, "schema.json"), encoding="utf-8") as f:
    schema = json.load(f)

with open(os.path.join(folder, "release_evidences.json"), encoding="utf-8-sig") as f:
    metadata = json.load(f)

questions = {}
for name in schema["features"]:
    questions[name] = metadata[name]["question_en"]

with open(os.path.join(folder, "questions.json"), "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2)